# Week 7 · One Agent Is Not Enough — Multi-Agent Systems

**GenAI & Agentic AI Programme · 30-minute lab** *(second half of today's session: the ADK
multi-agent lab on Google Skill Boost)*

---

Last week you built single agents and we deliberately deferred a problem: our change-risk
assessor both *investigated* and *recorded verdicts*. Pile every tool and responsibility onto
one agent and reliability degrades — the system prompt bloats, tool choice gets noisier,
and one confused step poisons everything downstream. The fix is the same one organisations
discovered long ago: **specialists, plus someone to coordinate them.**

The interesting question is the coordinator. Today, two competing answers in 30 minutes:

| | 🅐 CrewAI | 🅑 LangGraph *(LangChain's orchestration library)* |
|---|---|---|
| **Philosophy** | **Declare a team** — roles, goals, tasks; the framework runs the org chart | **Draw the flowchart** — typed shared state, nodes, edges; you own the control flow |
| **Scenario** | Product-launch **content studio**: researcher → copywriter → editor | **Spec-to-function pipeline**: coder ⇄ tester loop with a quality gate |
| **Model** | `gpt-4o-mini` | `claude-haiku-4-5` |
| **Shows off** | Role prompting, task chaining, sequential handoffs | Conditional edges, **cycles**, and a **non-LLM node** |

Two deliberately **non-ITSM** scenarios this week: the pattern is the lesson, not the domain.
(You'll map both patterns back onto IT operations in the exercises — and see Google's take,
ADK, in the Skill Boost lab next.)

Every individual agent below still runs Week 6's loop — LLM + tools + loop — internally.
Multi-agent is about what happens *between* the loops.

> ⏱ Pacing: setup 3 · primer 2 · 🅐 CrewAI 12 · 🅑 LangGraph 10 · wrap 3 ≈ 30 min

In [ ]:
# ── Setup: pinned installs (CrewAI is a heavy install — start this cell first, ~2 min) ──
%pip -q install "crewai==1.15.1" "langgraph==1.2.7" "langchain-anthropic==1.4.8"

import logging, warnings, os
warnings.filterwarnings("ignore")
os.environ["OTEL_SDK_DISABLED"] = "true"          # CrewAI telemetry off for the lab
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
print("Installs done.")

In [ ]:
# ── Display helpers (inlined — self-contained, same conventions as Week 6) ────
import json, os, textwrap
from getpass import getpass

W = 100
def _rule(ch="─"): print(ch * W)

def banner(title: str):
    print(); _rule("━"); print(f"  {title}"); _rule("━")

def section(title: str):
    print(); print(f"■ {title}"); _rule()

def observe(text: str):
    print(); print("👀 OBSERVE"); print(textwrap.fill(text, W))

def discuss(text: str):
    print(); print("💬 DISCUSS"); print(textwrap.fill(text, W))

def warn(text: str):
    print(); print("⚠ " + textwrap.fill(text, W - 3))

def success(text: str):
    print(); print("✔ " + textwrap.fill(text, W - 3))

def trace(kind: str, text: str):
    tag = {"node": "  [node]   ", "model": "  [model]  ",
           "call": "  → call   ", "result": "  ← result "}[kind]
    body = textwrap.fill(text, W - len(tag), subsequent_indent=" " * len(tag))
    print(tag + body)

def compare(blocks: dict):
    for label, text in blocks.items():
        print(); print(f"┌─ {label} " + "─" * max(0, W - len(label) - 4))
        print(textwrap.fill(str(text), W - 2, initial_indent="│ ", subsequent_indent="│ "))
    print("└" + "─" * (W - 1))

# ── The two keys you already have ─────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

OPENAI_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-haiku-4-5"

success("Helpers + auth ready.")

## Primer · two ways to coordinate specialists *(2 min)*

```
  🅐 DECLARE A TEAM (CrewAI)                🅑 DRAW THE FLOWCHART (LangGraph)

  researcher ──► copywriter ──► editor          ┌─────► coder ─────┐
   (role +        (role +       (role +         │                  ▼
    goal +         goal +        goal +       revise             tester   ◄── NOT an LLM —
    tools)         task ctx)     task ctx)      │                  │          plain Python
                                                └── fail ◄── pass ─┴──► ship
  an assembly line the framework runs         a state machine YOU design — with cycles
```

Same trade-off you met in Week 6, one level up. CrewAI is **ceremony-light and opinionated**:
describe *who* does *what*, get a sequential (or hierarchical) pipeline. LangGraph is
**control-heavy and unopinionated**: every node is just a function reading and writing a shared
typed state, and edges — including *conditional* ones and *loops* — are yours to wire. Note
what an assembly line cannot do: go **backwards**. Quality-gate-and-retry needs a cycle, and
that's exactly the demo for 🅑.

---
# 🅐 CrewAI · a product-launch content studio *(≈12 min)*

**Scenario:** marketing needs launch copy for **TrailMate**, an AI trekking-companion app.
Three specialists: a **market researcher** (the only one with tool access — an internal
product-brief lookup), a **copywriter**, and an **editor** who enforces the brand rules.

The CrewAI vocabulary maps onto Week 6 concepts one-to-one:

- **`Agent`** = a role-scoped system prompt (`role` + `goal` + `backstory`) plus optional tools —
  each agent runs its own LLM+tools+loop internally.
- **`Task`** = a unit of work with an `expected_output` contract, assigned to one agent.
- **`context=[...]`** = the handoff: a task receives the *outputs* of earlier tasks — the
  team-level version of Week 6's `messages` list.
- **`Crew` + `Process.sequential`** = the assembly line.

In [ ]:
# ── The researcher's tool: an internal product wiki (mock) ────────────────────
PRODUCT_BRIEFS = {
    "trailmate": {
        "product": "TrailMate",
        "one_liner": "An AI trekking companion app for Indian trails.",
        "key_features": ["offline AI route guidance for 400+ Indian trails",
                         "vitals-based pace coaching via smartwatch integration",
                         "SOS beacon with satellite fallback",
                         "trail-condition reports crowdsourced and AI-verified"],
        "target_audience": "urban professionals aged 24-40 taking weekend Himalayan and Western Ghats treks",
        "price": "₹499/year, first month free",
        "brand_rules": ["never promise safety outcomes, only assistance",
                        "no competitor names", "tone: confident, warm, zero hype-words like 'revolutionary'"],
    },
}

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

@tool("Product brief lookup")
def product_brief(product: str) -> str:
    """Return the internal launch brief for a product: features, audience, price, brand rules.

    Args:
        product: The product name, e.g. 'TrailMate'.
    """
    rec = PRODUCT_BRIEFS.get(product.strip().lower())
    if rec is None:
        return json.dumps({"status": "error",
                           "error_message": f"No brief for '{product}'. Known: {sorted(PRODUCT_BRIEFS)}"})
    return json.dumps({"status": "success", "brief": rec})

llm = LLM(model=OPENAI_MODEL)

researcher = Agent(
    role="Market Researcher",
    goal="Produce accurate, structured product facts for the writing team — never invent features.",
    backstory=("A meticulous researcher who trusts only the internal brief. If the brief "
               "doesn't say it, it isn't a fact."),
    llm=llm, tools=[product_brief], verbose=True,
)

copywriter = Agent(
    role="Launch Copywriter",
    goal="Turn research into a compelling ~150-word launch announcement.",
    backstory=("A consumer-app copywriter who writes for busy urban readers: concrete "
               "benefits, short sentences, one clear call to action."),
    llm=llm, verbose=True,
)

editor = Agent(
    role="Brand Editor",
    goal="Enforce brand rules and quality: factual, on-tone, within length.",
    backstory=("The last line of defence before publication. Ruthless about hype words, "
               "unverifiable claims, and anything violating the brand rules in the research."),
    llm=llm, verbose=True,
)

success("Three specialists defined. Note the asymmetry: ONLY the researcher holds a tool — "
        "the writers work from her output, not from the source system.")

In [ ]:
# ── Tasks: the work, the contract, and the handoffs ───────────────────────────
research_task = Task(
    description=("Research {product} using the product brief tool. Extract: one-liner, "
                 "key features, target audience, price, and the brand rules verbatim."),
    expected_output="A structured fact sheet with all five sections, no invented information.",
    agent=researcher,
)

writing_task = Task(
    description=("Write a launch announcement for {product} of roughly 150 words for the "
                 "target audience identified in the research. Lead with the strongest benefit, "
                 "cover at least three features, include the price, end with a call to action."),
    expected_output="A ~150-word launch announcement.",
    agent=copywriter,
    context=[research_task],                      # ← the handoff
)

editing_task = Task(
    description=("Edit the draft announcement against the brand rules from the research. "
                 "Fix violations, verify every claim against the fact sheet, keep it under "
                 "160 words. Output the final approved copy only."),
    expected_output="The final, publication-ready announcement under 160 words.",
    agent=editor,
    context=[research_task, writing_task],        # ← receives BOTH upstream outputs
)

content_studio = Crew(
    agents=[researcher, copywriter, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,                                  # CrewAI's own trace — watch the handoffs live
)

banner("CREW KICKOFF · TrailMate launch")
result = content_studio.kickoff(inputs={"product": "TrailMate"})

In [ ]:
section("Final deliverable (editor's output)")
print(result.raw)

section("Per-task outputs — the handoff artefacts")
for i, t_out in enumerate(result.tasks_output, 1):
    print(f"\n[{i}] {t_out.agent} → {textwrap.shorten(t_out.raw, 140)}")

observe("Scroll the verbose trace above: the researcher called the tool (Week 6's loop, alive "
        "inside one team member), the copywriter received the fact sheet as context — not the "
        "tool — and the editor received both. In Week 6 you hand-carried results between model "
        "calls via a messages list; context=[...] is that list, promoted to team level.")

discuss("Why does only the researcher get the tool? Least privilege — the same reason your "
        "executor agents will hold the dangerous write-tools next. And notice what this "
        "assembly line CANNOT express: if the editor hates the draft, there is no edge back "
        "to the copywriter. Sequential crews go forward only. For revise-until-good, you need "
        "a cycle — enter LangGraph.")

---
# 🅑 LangGraph · a spec-to-function pipeline with a quality gate *(≈10 min)*

**Scenario:** an engineering classic. A **coder** agent (Claude) writes a function from a
spec; a **tester** node runs real unit tests against the draft; failures route *back* to the
coder with the error report as feedback — up to 3 rounds, then ship or stop.

Three ideas, all new versus CrewAI:

1. **Shared typed state.** One `TypedDict` flows through the graph; every node reads it and
   returns just the fields it changes. No hidden context-passing — the state *is* the system.
2. **The tester is NOT an LLM.** It's deterministic Python running `assert`s. Graph nodes are
   just functions — mixing LLM judgment with hard checks is the whole point.
3. **Conditional edges make cycles.** `tester → coder` on failure, `tester → END` on pass.
   A framework built on graphs can express retry loops that assembly lines cannot.

> ⚠ The tester `exec`s model-written code. Fine for a lab (isolated namespace, trivial spec);
> in production, generated code runs in a **sandbox** — never in your process.

In [ ]:
import re
from typing import TypedDict
from langchain_anthropic import ChatAnthropic

coder_llm = ChatAnthropic(model=CLAUDE_MODEL, max_tokens=1500)

SPEC = ("Write a Python function  slugify(title: str) -> str  that converts a title into a "
        "URL-safe slug: lowercase, words separated by single hyphens, no leading or trailing "
        "hyphens, and only characters [a-z0-9-] in the output.")

# Hidden edge cases — the coder sees the SPEC, the tester knows the truth. Sound familiar?
TEST_CASES = [
    ("Hello, World!",                    "hello-world"),
    ("  Multiple   spaces  &  symbols!!","multiple-spaces-symbols"),
    ("Café Déjà Vu",                     "cafe-deja-vu"),          # unicode → ascii
    ("--Already--Slugged--",             "already-slugged"),
    ("2026: Year of Agents",             "2026-year-of-agents"),
]

class PipelineState(TypedDict):
    spec: str
    draft: str            # current code
    feedback: str         # tester's failure report
    rounds: int
    passed: bool

def _extract_code(text: str) -> str:
    m = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    return (m.group(1) if m else text).strip()

# ── Node 1: the coder (an LLM node) ───────────────────────────────────────────
def coder(state: PipelineState) -> dict:
    prompt = f"Task:\n{state['spec']}\n\nReply with ONLY a Python code block containing the function."
    if state["feedback"]:
        prompt += (f"\n\nYour previous attempt:\n```python\n{state['draft']}\n```\n"
                   f"It FAILED these tests:\n{state['feedback']}\n"
                   "Fix the function and resubmit it in full.")
    trace("node", f"coder · round {state['rounds'] + 1}"
                  + (" · revising against feedback" if state["feedback"] else " · first attempt"))
    reply = coder_llm.invoke(prompt)
    return {"draft": _extract_code(reply.content), "rounds": state["rounds"] + 1}

# ── Node 2: the tester (a DETERMINISTIC node — no LLM anywhere) ───────────────
def tester(state: PipelineState) -> dict:
    ns: dict = {}
    failures = []
    try:
        exec(state["draft"], ns)                     # lab-only; sandbox this in production
        fn = ns.get("slugify")
        if not callable(fn):
            failures.append("No function named slugify was defined.")
        else:
            for given, expected in TEST_CASES:
                try:
                    got = fn(given)
                    if got != expected:
                        failures.append(f"slugify({given!r}) → {got!r}, expected {expected!r}")
                except Exception as e:
                    failures.append(f"slugify({given!r}) raised {type(e).__name__}: {e}")
    except Exception as e:
        failures.append(f"Code failed to execute: {type(e).__name__}: {e}")
    passed = not failures
    trace("node", f"tester · {'ALL ' + str(len(TEST_CASES)) + ' TESTS PASS' if passed else str(len(failures)) + ' failure(s)'}")
    for f in failures:
        trace("result", f)
    return {"passed": passed, "feedback": "\n".join(failures)}

# ── The router: where control flow lives ─────────────────────────────────────
MAX_ROUNDS = 3
def route_after_tests(state: PipelineState) -> str:
    if state["passed"]:
        return "ship"
    if state["rounds"] >= MAX_ROUNDS:
        return "give_up"
    return "revise"

print("Nodes defined: coder (LLM), tester (plain Python), router (plain Python).")

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(PipelineState)
g.add_node("coder", coder)
g.add_node("tester", tester)
g.add_edge(START, "coder")
g.add_edge("coder", "tester")
g.add_conditional_edges("tester", route_after_tests,
                        {"revise": "coder", "ship": END, "give_up": END})
pipeline = g.compile()

#   START ─► coder ─► tester ─┬─ pass ────────► END (ship)
#              ▲              ├─ fail,<3 ─► back to coder
#              └──────────────┘
#                             └─ fail,≥3 ────► END (give up)

banner("PIPELINE RUN · " + SPEC[:60] + "…")
final = pipeline.invoke({"spec": SPEC, "draft": "", "feedback": "", "rounds": 0, "passed": False})

section(f"Outcome after {final['rounds']} round(s)")
if final["passed"]:
    success("Shipped — the draft below passed all hidden tests.")
else:
    warn("Gave up after MAX_ROUNDS — final draft below still fails; see feedback above.")
print("\n" + final["draft"])

### Observe & discuss · 🅑

If the trace shows **more than one coder round**, you watched the cycle earn its keep: the
tester's failure report (most likely the unicode case — `Café Déjà Vu` — which the spec never
mentioned) became the coder's feedback, and the revision passed. If the coder nailed it in
round 1, rerun the cell a couple of times or add a nastier test case — the *architecture* is
the point: failures had somewhere to go other than your lap.

Three discussion beats:

1. **The gate is deterministic.** An LLM critic can be argued with; `assert` cannot. Put hard
   checks in plain-Python nodes and reserve LLM judgment for what genuinely needs it —
   the same intelligence-vs-safety split as Week 6's tool guard rails, now drawn on the map.
2. **The hidden tests are the real world.** Specs are always incomplete; production traffic is
   the tester that knows edge cases you didn't write down. A revise loop is how a system
   absorbs that gap automatically.
3. **`MAX_ROUNDS` is Week 6's `max_iterations`,** one level up. Cycles without a hard stop are
   outages waiting to happen. Your runtime owns termination — still.

In [ ]:
compare({
    "🅐 CrewAI · declare a team":
        "Roles + tasks + Process.sequential. Fastest path to a team of specialists; handoffs "
        "via task context; built-in trace. Opinionated: control flow belongs to the framework "
        "— forward-only in sequential mode. Reach for it when the work IS a pipeline of "
        "role-shaped steps.",
    "🅑 LangGraph · draw the flowchart":
        "Typed shared state + nodes (LLM or plain Python) + edges you wire yourself, including "
        "conditionals and cycles. Maximum control, more ceremony. Reach for it when control "
        "flow IS the design: retries, quality gates, branches, human checkpoints.",
    "Week 6 · the loop, for perspective":
        "Every agent above still runs LLM+tools+loop inside. CrewAI hides it per-agent; "
        "LangGraph hides it per-node. Multi-agent frameworks orchestrate BETWEEN loops — "
        "nothing about the loop itself changed.",
})

discuss("The two philosophies aren't rivals so much as layers: plenty of production systems "
        "draw the outer control flow as a graph and drop a crew inside one node. And Google's "
        "ADK — your Skill Boost lab, next — answers the same question with sub-agents and "
        "delegation. Same trade-offs, third vocabulary: you now have the map to read it.")

## Production notes & bridges

Deferred on purpose, named so they don't vanish:

1. **Humans in the loop.** Both frameworks have first-class hooks — CrewAI's `human_input=True`
   pauses a task for approval; LangGraph's **interrupts** make "a human" just another node the
   graph waits on. The editor and the tester in today's lab are *automated* reviewers; wiring
   a human into those same gate positions is the production upgrade.
2. **Hierarchy & delegation.** Today's coordination was sequential and graph-shaped. Both
   frameworks also offer a manager pattern (CrewAI `Process.hierarchical`, LangGraph
   supervisor graphs) where a coordinator agent *decides* who works next — powerful, and
   harder to audit. Start flat; earn hierarchy.
3. **Audit across agents.** Week 6's audit log tracked one agent's writes. With a team, the
   trail must answer "which agent, in which task, on whose upstream output?" — provenance,
   not just actions.

**Next 30 minutes:** the ADK multi-agent lab on Skill Boost — Google's answer to today's
coordinator question, with sub-agents and delegation on the framework you'll also meet again
in production GCP work.

**Week 8 (finale):** every team today shared tools by *importing functions*. That dies the
moment agents live in different processes, teams, or companies. MCP makes tools — and the
context around them — a standardised, discoverable service any agent can attach to.

## Exercise (take-home)

- **🅐 CrewAI:** add a fourth specialist — a *Social Media Writer* whose task takes
  `context=[editing_task]` and produces three platform-ready posts (X, LinkedIn, Instagram)
  from the approved copy. One agent, one task, one list edit: notice how cheap growing a
  sequential crew is.
- **🅑 LangGraph:** add a `security_review` node between a passing test and `END`: an LLM node
  that rejects drafts containing `eval`, `exec`, `os.system`, or file I/O, routing rejections
  back to the coder with the reason. You've just built a two-gate pipeline — one deterministic,
  one judgment-based.
- **Stretch (back to our world):** rebuild Week 6's SLA watchdog as a two-agent CrewAI team —
  an *analyst* (read-only tools) and an *executor* (holds `escalate_incident`, acts only on
  the analyst's findings). Least privilege, expressed as an org chart.